<a href="https://colab.research.google.com/github/thedarredondo/data-science-fundamentals/blob/main/FastTrack/IntroMultLinSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiple Linear Modeling,
. . . but way oversimplified.

For a less simplified treatment, you can look at my [Intro to Bayesian Statistics course](https://github.com/thedarredondo/data-science-fundamentals)

This jupyter notebook will walk you through how to create, read, and interpret Bayesian multiple linear models from a causal perspective using the bambi library.

In more readable terms: We will learn the simplest (good) way to statitically argue that something causes something else. None of this may feel especially simple, but I promise this is simplest things can get, while still allowing us to make interesting statements about the world.

Here's the roadmap for how we'll gain the ability to make interesting statistical claims:

1. Data Wrangling: A brief description of how to load data into a jupyter notebook in the right format.

2. Causal Diagrams: A brief example of how to phrase a network of hypotheses from a causal perspective, using graphs

3. Making a Linear Model: A walkthrough of how to make the simplest (good) prediction engine. This can be broken up into sub parts:
    - Constructing a model with a single estimator
    - Making a model with multiple estimators
    - Interpreting models probalistically
    - Comparing models and making Causal interpretations

This little explainer will involve "coding", and it will involve thinking hard. With a can-do attitude and an open mind, anyone is capable of doing both. You have the oppurtunity to learn some powerful skills and techniques--but more importantly, you have the opportunity to learn a new way to look at the world.

#### Packages for Python Programming

Below are your first lines of python code. Run them by pressing the "play" button in the left hand side of the code blocks.

The first one will spit out a bunch of text, the second one will seem to do nothing.

Don't worry about any of it; they are neccesary to load in the tools we will you to make our statistical models. Loading in tools that someone else made is a common first step when coding with python--so common, that most people don't even mention it.

In [ ]:
!pip install bambi

In [ ]:
from google.colab import files
from google.colab.output import eval_js
import io
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import graphviz as gv
import arviz as az
import pymc as pm
import bambi as bmb


Note that, once you've run a code block, you can click on the three dots right below the run button to pull up the option to delete that massive bit of text. Or you can click the little down arrow to temporaily hide it.

##  Data Wrangling

We will focus on one way to load properly formatted data into a jupyter notebook. After that, I will explain many ways in which things can be more complicated than our cushy example

### Loading Data

We're going to learn how to load data into a jupyter notebook.

To start, I will assume a passing familiarity with spreadsheet-like tools such as google sheets--such as [this google sheet of bodyfat data](https://docs.google.com/spreadsheets/d/1HQB6XRhp49VMd81OmZS_qDbSDecPl6z05TmNLO4mgDs/edit?usp=sharing).

We will now transfer that data from the google sheet, to your computer, and then to a dataframe in colab.

1. Go ahead and click on that link, or otherwise navigate to that google sheet
2. Click File, then Download, and  then Comma Sperated Value (csv).
3. Now, run the code block below.
4. Click the "Choose Files" button, and then select the file you just downloaded (this should be in your downloads folder), and then upload the file.
5. Run the subsequent code block. Make sure that the name of the file--in bold below where you uploaded it--is inside the red quotation marks in the relevant code block.
6. Run the code block that just sayds "body". If that outputs a table that looks the original csv, then congrajulation! You have successfully uploaded the data.

In [ ]:
uploaded = files.upload()

In [ ]:
body = pd.read_csv(io.BytesIO(uploaded['REPLACE ME']))

In [ ]:
body

In [ ]:
#If you really can't get the above thing to work, you can remove the hastag below and run this line
#body = pd.read_csv('https://raw.githubusercontent.com/thedarredondo/data-science-fundamentals/refs/heads/main/Data/body_fat.csv')

### "Clean" Data

If we want to run any type of statistical model, we need data, and that data needs to be in an appropriate format. Almost always, the easiest format to work with is a .csv file ( which stands for comma separated format). Note that the data you just uploaded is a csv, that we were able to covert into something called a dataframe.

#### Columns

The columns of our table (or csv) signify variables, or the class of thing we are measuring.
For example, siri signifies a body fat percentage measurment for people. I know we have measurements of 12.3, 6.1 and 25.3, because those numbers are beneath the column name "siri".
Note that I didn't specify the units. We don't include units in the data we will feed to our statistical model. Instead, you store units in your head, in a separate document, or in the column name.

#### Rows

The rows of our table signify observations belonging to one individual/case/instance. In the table above, a row is a specific person. The person corresponding to row zero has a body fat measurement of 12.3, an age of 23, a weigth of 70.1 (kilograms?), a height of 172 (cm?), etc.

#### Comments on Data Formatting and Data Types

The fastest way to make your life hard is to format your data differently than I outlined above.
If your data is organized differently than my example, the first step is to cry.
The second step is to search “how to convert [description of your messy data] into a csv”, and get to work.

Also: we can have non-numerical data in our table/csv. For example, we could have the variable “gender”, which wouldn’t be a number. If we are careful with the way we've inputed such data, then our model will handle the data without any problems--so long as that data is an estimator with at least one other quantiative estimator. We can also make models that predict categorical variables, but I won't be explicity covering them in this notebook. The method below can predict categorical data with a different "family" setting--but using that "family" setting correctly involves a depth of understanding that I am avoiding at this point.



## Causal Inference

As concious, curious people, we want to know Why.

As in:
- Why do octopus beaks weigh as much as them do?
- Why do people rent bikes?
- Why do citrus fruits cure scurvy?

We can answer all these questions (and more!) using the power of Causal Inference.

### An Interlude on Causality through Citrus

When I say A causes B, I am making a mathematical statement: I say that A is a variable in a formula for the function that defines B.

Example: Citrus fruits have Vitamin C (asorbic acid). Humans do not produce asorbic acid, but we need it as a reagent in many vital bodily chemcial processes. Without Vitamin C, those processes do not occur, resulting in various symptoms. Those symptoms have a name: scurvy. (This example comes from [The Book of Why](https://bayes.cs.ucla.edu/WHY/))

I can summarize the functional connection between various "things" (variables) by using a a Directed Acyclic Graph (DAG). Note that Causal DAGs are also called Structural Causal Models (SCMs).

In [ ]:
#Run this code block to see the DAG/SCM for Scurvy

s_dag = gv.Digraph(name="Scurvy DAG")

s_dag.node('C','No Citrus Fruit')
s_dag.node('A','No Asorbic Acid')
s_dag.node('B','Failed Bodiliy Processes')
s_dag.node('S','Scurvy')

s_dag.edges(['CA','AB','BS',])

s_dag

This level of detail might be a bit of overkill; perhaps we could use the simpler one below

In [ ]:
#Run this code block to see the DAG/SCM for Scurvy

c_dag = gv.Digraph(name="Citrus DAG")

c_dag.node('C','No Citrus Fruit')
c_dag.node('S','Scurvy')

c_dag.edges(['CS',])

c_dag

. . . but perhaps that's too much simplification.

After all, I can get asorbic acid from fresh meat, as did some Sailors on the 1875 British Arctic Expedition, and not recieve asorbic acid from boiled lime juice (boiling chemcially decomposes the molecule).

That incident caused many in the scientific community to decide that well prepared meat was a better preventative for scurvy. Which caused many to die on the Jackson-Harmsworth Expedition to the Arctic in 1894, and on both Robert Falcon Scott Expeditions to the Antarctic in 1903 and 1911.

Those misguided scientists unwittingly subscribed to the following set of hypotheses:

In [ ]:
fs_dag = gv.Digraph(name="Fake Scurvy DAG")

fs_dag.node('C','No Citrus Fruit')
fs_dag.node('M',"Meat")
fs_dag.node('P',"'Contamination' in meat")
fs_dag.node('B','Failed Bodiliy Processes')
fs_dag.node('S','Scurvy')

fs_dag.edges(['PB','BS','MP',])

fs_dag

Note that I said "set of hypotheses". A Causal Diagram is a set of hypotheses: every variable (node, or oval) is a hypothesis, every arrow is a hypohthesis, every *absence* of an arrow is a hypothesis, and every subset  of those things
(including the entire diagram itself) is a hypothesis.

Examples:
- 19th century scientists hypothesized that citrus fruits did not help with scurvy. Therefore, there is no arrow from citrus fruit to scurvy in the above diagram; the lack of an arrow means those variables are independent.
- Likewise, those scientists believed that "contamination" in meat causes bodily processes to fail (hypothesis), those failed bodily processes manifest as the disease called scurvy (hypothesis), and that removing any "taint" from meat will break the chain of causal events that lead to scurvy (hypothesis).

These hypothesises were refuted by later evidence: all the people on expeditions who died following believing in those hypothesises.

So perhaps the below graph, with our modern chemical understanding, is better.

In [ ]:
s1_dag = gv.Digraph(name="Scurvy1 DAG")

s1_dag.node('C','No Citrus Fruit')
s1_dag.node('A','No Asorbic Acid')
s1_dag.node('S','Scurvy')

s1_dag.edges(['CA','AS',])

s1_dag

The above causal diagram leaves out fresh meat, and other sources of asorbic acid (vitamin C), but it communicates the main idea: citrus fruits prevent scurvy through their asorbic acid content.
With that nuance, people can make better decisions, like packing vitmain C pills on expeditions, instead of bulky fruit.

Causal diagrams help us organize our thoughts and ideas into logically consistent and easy to inerpret statements. In other words, they mathamatize our ideas; they turn our guesses into hypotheses.

### Making Causal Assumptions

When making a causal diagram, we often run into a problem: we might not be confident in our guesses, in our causal assumptions. When (not if, when) we encounter a scenario we aren't confident with, we can use other information to help us draw our causal diagram, to formalize our causal assumptions.

#### Pair Plots

Pair plots are one of the most common and most useful tools for making a rough-draft causal diagram.

This is where the intuition you built in learning about simple linear regression will come into play.

Describe the relationships in as many of the scatterplots below as you can handle.
Also, answer this question: are the conditions for simple linear regression met for any of these scatter plots?

In [ ]:
sns.pairplot(body)

To answer the question: most of them!

The plots with age seem the weakest, and there are a couple of other plots with weak trends. But for the most part, the plots have moderate to strong positive, linear associations.

That means I should suspect some sort of causal relationship between these variables. I'll draw a causal diagram to define my hypotheses.

In [ ]:
bodydag = gv.Digraph(name="bodyDAG")

bodydag.node('W','weight')
bodydag.node('H','height')
bodydag.node('F','siri/body fat')
bodydag.node('A','abdomen')
bodydag.node('R','wrist')
bodydag.node('T','thigh')
bodydag.node('O','age')

bodydag.edges(['HW','HR','HA','HT','FA','FT','FR','FW','AW','RW','TW','OF'])

bodydag

Note that, even though there were trends between almost all the variables, I didn't put every possible arrow. **My goal is to represent what I think is true in reality, not to copy what I see in the pair plot.**

For example, wrist, thigh, and abdomen all have positive associations with one another, but I didn't draw any arrows direct arrows between them. This signifies my belief that those particular trends in the pair plot are "spurious", or caused by some type of confounding. In particular, I think two of the confounding variables are in the data set: height, and body fat percentage (siri). Actually, I think there's another one for thigh--muscle mass percentage--but its not in our data set, so I decided not to depict it.


With that, we now have a well defined set of hypotheses about our data. You might have a different set of hypotheses than me, a different causal diagram. Your diagram might even be "better".

We'll get to find once we learn how to make, and evalute, linear models.

### From a Causal Model to Linear Model

We have causal hypotheses, and we have data. Now, we will work on bring them together in the same object.

There are many valid mathematical objects we can use to bring our hypotheses and data together; the "simplest", "good" method is to create a linear model.

#### Scary Algebra Notation for Linear Models

I've found that people don't believe me when I claim that causal diagrams are rigourous mathematical objects--so I've included this section with algebra notion.

This is a common notation for generlaized linear models that you will come across if you study these techniques further. I am not asking you to fully embrace the notation below, but I am asking you to push yourself to read it.

If I only wanted to predict weight with height, then I could write a  linear model using the following:

$W_{i} ∼ Normal(μ_{i},σ)$ [my likelihood, or how weight is distributed]

$μ_{i}= α + β_{H}H_{i}$ [linear model, or how the mean of toWt is distributied]

$α ∼ HalfNormal(10)$ [prior for intercept]

$β_{H} ∼ Normal(0,10)$ [prior for slope with respect to height]

$σ ∼ HalfNormal(10)$ [prior for stdev of weight]

The above would correspond to the following causal diagram:

In [ ]:
minidag = gv.Digraph(name="miniDAG")

minidag.node('W','weight')
minidag.node('H','height')

minidag.edges(['HW',])

minidag

Here's a causal diagram with height and siri/body fat percentage:

In [ ]:
bodydag = gv.Digraph(name="bodyDAG")

bodydag.node('W','weight')
bodydag.node('H','height')
bodydag.node('F','siri/body fat')


bodydag.edges(['HW','FW',])

bodydag

And this is what that diagram looks like in algebra form:

$W_{i} ∼ Normal(μ_{i},σ)$ [my likelihood, or how totWt is distributed]

$μ_{i}= α + β_{H}H_{i} + β_{F}F_{i}$ [linear model, or how the mean of toWt is distributied]

$α ∼ HalfNormal(10)$ [prior for intercept]

$β_{H} ∼ Normal(0,10)$ [prior for slope with respect to height]

$β_{F} ∼ Normal(0,10)$ [prior for slope with respect to siri]

$σ ∼ HalfNormal(10)$ [prior for stdev of weight]

Now for the actual full depiction of my hypotheses--minus age, since that doesn't directly affect weight (the thing I'm trying to predict) and its influnce is small.

In [ ]:
bodydag = gv.Digraph(name="bodyDAG")

bodydag.node('W','weight')
bodydag.node('H','height')
bodydag.node('F','siri/body fat')
bodydag.node('A','abdomen')
bodydag.node('R','wrist')
bodydag.node('T','thigh')

bodydag.edges(['HW','HR','HA','HT','FA','FT','FR','FW','AW','RW','TW',])

bodydag

And those hypotheses in algebra speak:

$T_{i} ∼ Normal(μ_{i},σ)$

$μ_{i}= α + β_{H}H_{i} + β_{F}F_{i} + β_{A}A_{i} + β_{R}R_{i} + β_{T}T_{i} + β_{HA}H_{i}A_{i} + β_{HR}H_{i}R_{i}+ β_{HT}H_{i}T_{i} + β_{FA}F_{i}A_{i} + β_{FR}F_{i}R_{i} + β_{FT}F_{i}T_{i}$

$α ∼ Normal(80.9984, 269.0719)$

$β_{H} ∼ Normal(0,4.6075)$

$β_{F} ∼ Normal(0,3.6903)$

$β_{A} ∼ Normal(0,3.0069)$ [prior for slope of abdomen]

$β_{R} ∼ Normal(0,33.6206)$ [wrist]

$β_{T} ∼ Normal(0,6.1997)$ [thigh]

$β_{HA} ∼ Normal(0,0.0152)$ [prior for slope with respect to interaction between height and abdomen]

$β_{HR} ∼ Normal(0,0.1287)$ [height and wrist]

$β_{HT} ∼ Normal(0,0.0283)$ [height and thigh]

$β_{FA} ∼ Normal(0,0.032)$ [prior for slope with respect to interaction between siri and abdomen]

$β_{FR} ∼ Normal(0,0.0.194)$ [siri and wrist]

$β_{FT} ∼ Normal(0,0.0555)$ [siri and thigh]

$σ ∼ HalfStudentT(4, 12.2621)$

If this all seems overwhelming: that's ok! I never explained that $\sim$ means "distributed as a", nor have I explained what a prior or a likelihood are.

It takes a whole course to explain those things, so instead, we will use the short hand for linear models.

Example: the above linear model is often written as follows:

$T_{i} ∼ α + β_{H}H_{i} + β_{F}F_{i} + β_{A}A_{i} + β_{R}R_{i} + β_{T}T_{i} + β_{HA}H_{i}A_{i} + β_{HR}H_{i}R_{i}+ β_{HT}H_{i}T_{i} + β_{FA}F_{i}A_{i} + β_{FR}F_{i}R_{i} + β_{FT}F_{i}T_{i}$

We can make it even more simpe if we assume everything leave the slope and intercept terms hidden. Let's leave out the $i$s too, since I haven't explained what those mean.

$T ∼ H + F + A + R + T + HA + HR + HT + FA + FR + FT$

That might be too simple; instead of single letters we can use the variable names from the dataframe where our data is stored.

**weight ∼ height + siri + abdomen + wrist + thigh + height:abdomen + height:wrist + height:thigh + siri:abdomen + siri:wrist + siri:thigh**

The above is how we will write our model into code going forward.

Code?

Yes, code: statistics requires data, and data lives on computers, and code is how I tell computers to do things

#### Writing a Linear Model in Code

There are a number of ways to write a linear model in code, and they depond on the method and tool we choose.

I will make those choices for you:
- Our method: a Bayesian Generalized Linear Model (glm) with weak priors based on the data.
- Our tool: [bambi](https://bambinos.github.io/bambi/)



To turn my causal diagram into a bambi linear model, I write the following:

In [ ]:
model_body = bmb.Model("weight ~ height + siri + abdomen + wrist + thigh + height:abdomen + height:wrist + height:thigh + siri:abdomen + siri:wrist + siri:thigh", data = body)

I can print the model to see what it looks like. Notice that it's pretty similar to the scarcy algebra notation from before, and it's also where all those numbers are from.

In [ ]:
model_body

       Formula: weight ~ height + siri + abdomen + wrist + thigh + height:abdomen + height:wrist + height:thigh + siri:abdomen + siri:wrist + siri:thigh
        Family: gaussian
          Link: mu = identity
  Observations: 251
        Priors: 
    target = mu
        Common-level effects
            Intercept ~ Normal(mu: 80.9984, sigma: 1269.0719)
            height ~ Normal(mu: 0.0, sigma: 4.6057)
            siri ~ Normal(mu: 0.0, sigma: 3.6903)
            abdomen ~ Normal(mu: 0.0, sigma: 3.0069)
            wrist ~ Normal(mu: 0.0, sigma: 33.6206)
            thigh ~ Normal(mu: 0.0, sigma: 6.1997)
            height:abdomen ~ Normal(mu: 0.0, sigma: 0.0152)
            height:wrist ~ Normal(mu: 0.0, sigma: 0.1287)
            height:thigh ~ Normal(mu: 0.0, sigma: 0.0283)
            siri:abdomen ~ Normal(mu: 0.0, sigma: 0.032)
            siri:wrist ~ Normal(mu: 0.0, sigma: 0.194)
            siri:thigh ~ Normal(mu: 0.0, sigma: 0.0555)
        
        Auxiliary parameters
        

Now that my model definition exists in code, I need to give my model time to learn from data, and then generate predictions.

Both "learn from data" and "generate predcitions" might sound foreign to you. This is because both are more or less outlawed in frequentist statistics; frequentist statistics is what you learn in AP Stats or a typical college Intro to Stats class.

There is a whole lot going on behind the phrases "learn from data" and "generate predcitions" (and behind the next line of code, for that matter)--but we're not gonna worry about that.

Just take my word that the following line of code lets our model learn from and predict data. (. . . or read throught my [intro to Bayesian Statistics course](https://github.com/thedarredondo/data-science-fundamentals))

In [ ]:
idata_body = model_body.fit(idata_kwargs={"log_likelihood":True})

All the learning and predictions are now store in ```idata_body```. Which is great for the computer, but we need more.

Why?

Remember when I said causal diagrams are hypotheses? With our trained model, we can assess our hypotheses agaisnt our data/evidence.

#### Evaluating and Interpreting a single model: Posterior Predictive Checks

The first evaluation method for our model is a posterior predictive check, which is the visualization in the following two code blocks.

The "observed line" is like a smoothed histogram of the weight data. We want our model to "draw" that observed curve using the blue spagetti-like lines. The dotted orange line is the average of those thin blue lines.

In this case, our model does a great job! We might be able to do a little better with a t distribution of a skew normal, but this is defintely good enough--and possibly better.

In [ ]:
model_body.predict(idata_body, kind="response")

In [ ]:
az.plot_ppc(idata_body)

The second type of check requires our model to have at least two variables; we have six plus several interactions.

Unhelpfully, the below visualization is also called a posterior predictive check, so I'll call it a "scatterplot posterior predictive check" to help differentiate.

We want this plot to have a sensible line of best fit draw through the points, and we want the error bars (blue shading) to cover most of the points.

The line of best fit looks perfectly reasonable, but the shading is a little suspect. I think a t distribution in our likelihood would probably fix the issue.

Regardless, this is good enough.

In [ ]:
bmb.interpret.plot_predictions(model_body, idata_body, ["height"], pps=True, prob=0.998)
plt.scatter(body.height, body.weight)

At this point, this might not seem any different from running linear regression with height predicting weight in desmos. First, it is different, because this line is taking the other variables (bodyfat, wrist, abdomen, thigh) into account.

But I think you need a more viseral reason to belive they are different, which is why I've made the below visualization. It depects both bodyfat (siri) and abdomen predicting weight, "at the same time."

In [ ]:
bmb.interpret.plot_predictions(model_body, idata_body, ["siri","abdomen"], pps=True, prob=0.998, legend=False)
plt.scatter(body.siri, body.weight)

We can interpret the above plot as follows: as abdomen increaes in size, the prediction of weight by overall bodyfat (siri) goes from negative to positive association.

In other words, in people with smaller abdomens, more bodyfat means *less* weight, while in people with larger abdomens, more bodyfat means *more* weight.

Any hypotheses for why that might be true? Or is our model or data wrong?

Why our model could very well be wrong, and the data could have been collected poorly, I still have an explanation: we are missing the confounding variable "gender at birth". Female people tend to have higher bodyfat and smaller proportions than male people. We could test my hypothesis by recollecting the data with gender at birth included, and then rerun the model.

In this way, we can use our model to generate new hypotheses.

(the below graph is the same as above, just with the awkward legend that justifies my statement about abdomen size.)

In [ ]:
bmb.interpret.plot_predictions(model_body, idata_body, ["siri","abdomen"], pps=True, prob=0.998, legend=True)
plt.scatter(body.siri, body.weight)

### Evaluating Multiple Models, or "Testing Hypotheses"

We don't have to stop at evaluating one model, or one set of hypotheses; we can make many models and compare their general "performance".

How? All we have to do is make the models we want to compare, then put their output into a compare function.

(Once again, there is a lot going on behind the scenes. Technically we should do a posterior predictive check for every model AND do what I'm about to show you. But I'm skipping steps to give you the general idea.)

In [ ]:
model_nointer = bmb.Model("weight ~ height + siri + abdomen + wrist + thigh", data = body)
idata_nointer = model_nointer.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
model_onlyhs = bmb.Model("weight ~ height + siri", data = body)
idata_onlyhs = model_onlyhs.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
model_noh = bmb.Model("weight ~ siri + abdomen + wrist + thigh + siri:abdomen + siri:wrist + siri:thigh", data = body)
idata_noh = model_noh.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
model_nos = bmb.Model("weight ~ height + abdomen + wrist + thigh + height:abdomen + height:wrist + height:thigh", data = body)
idata_nos = model_nos.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
model_noa = bmb.Model("weight ~ height + siri + wrist + thigh + height:wrist + height:thigh + siri:wrist + siri:thigh", data = body)
idata_noa = model_noa.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
model_plusheightsiri = bmb.Model("weight ~ height + siri + abdomen + wrist + thigh + height:siri + height:abdomen + height:wrist + height:thigh + siri:abdomen + siri:wrist + siri:thigh", data = body)
idata_plusheightsiri = model_plusheightsiri.fit(idata_kwargs={"log_likelihood":True})

In [ ]:
cmp_df = az.compare({"My Hyp.":idata_body,"No Inter":idata_nointer,"Only H S":idata_onlyhs,"No Height":idata_noh,"No Siri":idata_nos,"No A":idata_noa, "+HghtSiri":idata_plusheightsiri})

In [ ]:
cmp_df

In [ ]:
az.plot_compare(cmp_df)

Focus on the above visualization.

This plot compares the elpd_loo (a one number summary of a posterior predictive check) of each model.

My first model is labeled "My Hyp.", which stands for "My Hypotheses"; the others are named after how they differ from my causal diagram inspired model.

Removing height and abdomen caused a massive decrease in predictive power, and only including height and bodyfat (siri) was even worse than that.

Interestingly, removing the interactions did not make much a difference. Same with removing bodyfat.

How do I know the plot tells me these things?

The open dot is the precise elpd_loo value for each model (higher is better), and the error bars around it are an estimate of the standard deviation of that elpd_loo.

My general rule of thumb, is that if the error bars overlap, then its reasonable to argue that the two models are functionally the same, performance-wise. If the error bars overlap with another model's dot, then it's strong evidence they have equivalent predictive power.
If the error bars do not overlap at all, then I have at least some evidence the models have different performance.

**These are imprecise guidelines.** When I make a decision about which model is "better" or "best", I should ALWAYS take context into account.

For example, even though there's evidence that the no interactions model is performatively equivalent to my hypotheses, I still read this as providing some evidence that my conception of the interactions is correct.

Likewise, I doubt that adding in an interaction between height and siri (bodyfat) is all that helpful, despite the fact that adding it techically increased my model's performance. Adding an additional estimator almost always increases model performance, so I will make myself have a bias agaisnt models with lots of estimators.

Hopefully, you'll notice that this means I'm picking my own model as "best"--and hopefully you've identified that as bias.

This is why we need peer review in science: it is easy to continue justifying my own ideas.

I've purposefully left this analysis undone; I would actually continue to test more predictor combinations, and I would experiment with a different likelihood, like the t distribution.

What would you do? What new variable combinations would you try? What data would you like to collect to add into the analysis? What would your causal diagram look like, if you could gather any relevant information?

These are the questions that should guide you when you read any scientific article/analysis, and when you plan an execute a scientific analysis of your own.

## Summary

We covered:
- what data storage format works best for statistical analysis, and one way to load that data into a jupyter notbook
- how to define and visualize causal hypotheses
- how to evaluate those hypotheses using generalized linear models (GLMs)

Minus collecting data (which is big deal, by the way), you now have the barest blueprint for how to "do science". Enjoy!
